# 生存分析模型比较实验
## 36组数据 × 4种方法

In [1]:
import numpy as np
import pandas as pd
np.random.seed(2026)

import warnings
warnings.filterwarnings('ignore')
import importlib
import config
importlib.reload(config)
from config import *
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = ['Heiti TC']
plt.rcParams['axes.unicode_minus'] = False

In [2]:
# 生成36组数据
random_seed = 2026
sample_sizes = [100, 500, 2000]
cens_lambdas = [0.1, 0.3, 0.5]
p_list = [1, 50]
hetero_list = [False, True]

datasets = []
configs = []

for n in sample_sizes:
    for cens in cens_lambdas:
        for p in p_list:
            for hetero in hetero_list:
                X, surv, time, event, real_censor = generate_weibull_data(
                    n=n, p=p, hetero=hetero, cens_rate=cens
                )
                datasets.append((X, surv, time, event))
                configs.append({
                    'n': n, 'cens_rate': cens, 'p': p, 
                    'hetero': hetero, 'real_censor': real_censor
                })

print(f'✅ 生成完成：{len(datasets)}组数据')

✅ 生成完成：36组数据


In [3]:
# 批量拟合四种模型
results = []

for i, (X, surv, time, event) in enumerate(datasets):
    cfg = configs[i]
    print(f"第 {i+1} 组: n={cfg['n']}, p={cfg['p']}, cens={cfg['cens_rate']}, hetero={cfg['hetero']}")

    # 1. Kaplan-Meier
    kmf = fit_kaplan_meier(time, event)
    
    # 2. Cox
    cox = fit_cox(X, time, event)
    c_cox = evaluate_model(cox, X, time, event, 'cox')
    
    # 3. Weibull
    weibull = fit_weibull(X, time, event)
    c_weibull = evaluate_model(weibull, X, time, event, 'weibull')
    
    # 4. 随机生存森林
    rsf = fit_rsf(X, time, event)
    c_rsf = evaluate_model(rsf, X, time, event, 'rsf')
    
    results.append({
        **cfg,
        'C_Cox': c_cox,
        'C_Weibull': c_weibull,
        'C_RSF': c_rsf
    })
print("数据处理完成")
df_results = pd.DataFrame(results)
df_results

第 1 组: n=100, p=1, cens=0.1, hetero=False
第 2 组: n=100, p=1, cens=0.1, hetero=True
第 3 组: n=100, p=50, cens=0.1, hetero=False
第 4 组: n=100, p=50, cens=0.1, hetero=True
第 5 组: n=100, p=1, cens=0.3, hetero=False
第 6 组: n=100, p=1, cens=0.3, hetero=True
第 7 组: n=100, p=50, cens=0.3, hetero=False
第 8 组: n=100, p=50, cens=0.3, hetero=True
第 9 组: n=100, p=1, cens=0.5, hetero=False
第 10 组: n=100, p=1, cens=0.5, hetero=True
第 11 组: n=100, p=50, cens=0.5, hetero=False
第 12 组: n=100, p=50, cens=0.5, hetero=True
第 13 组: n=500, p=1, cens=0.1, hetero=False
第 14 组: n=500, p=1, cens=0.1, hetero=True
第 15 组: n=500, p=50, cens=0.1, hetero=False
第 16 组: n=500, p=50, cens=0.1, hetero=True
第 17 组: n=500, p=1, cens=0.3, hetero=False
第 18 组: n=500, p=1, cens=0.3, hetero=True
第 19 组: n=500, p=50, cens=0.3, hetero=False
第 20 组: n=500, p=50, cens=0.3, hetero=True
第 21 组: n=500, p=1, cens=0.5, hetero=False
第 22 组: n=500, p=1, cens=0.5, hetero=True
第 23 组: n=500, p=50, cens=0.5, hetero=False
第 24 组: n=500, p=50,

,n,cens_rate,p,hetero,real_censor,C_Cox,C_Weibull,C_RSF
0,100,0.1,1,False,0.1000,0.454302,0.545698,0.741614
1,100,0.1,1,True,0.1500,0.490002,0.509998,0.764749
2,100,0.1,50,False,0.1000,0.107212,0.880225,0.948235
3,100,0.1,50,True,0.1400,0.124402,0.843472,0.956482
4,100,0.3,1,False,0.3600,0.432674,0.567326,0.752331
5,100,0.3,1,True,0.3000,0.423130,0.576870,0.727839
6,100,0.3,50,False,0.2800,0.100481,0.884286,0.956173
7,100,0.3,50,True,0.3500,0.078506,0.907959,0.955604
8,100,0.5,1,False,0.4800,0.533597,0.533597,0.809536
9,100,0.5,1,True,0.3400,0.517918,0.482082,0.731988


In [4]:
# 保存结果
df_results.to_csv('模型拟合结果.csv', index=False)
print('✅ 结果已保存')

✅ 结果已保存


## 1. 模拟数据上的模型拟合效果总结

### 1.1 整体模型性能排序（按C-index）

#### 平均 C-index
- **RSF (点预测准确性最佳)**：0.8561
- **Weibull AFT**：0.6871
- **Cox PHF**：0.3108
- **KM**：无法计算 C-index（非参数模型，无个体风险评分）

**关键结论**：非参数/树基模型（RSF）在 Weibull 数据生成的设置下，预测准确性显著优于半参数（Cox）和参数（Weibull）。

---

### 1.2 模型性能按样本量的表现

| 样本量 | Cox (C-index) | Weibull | RSF |
|-------|---------------|---------|-----|
| n=100 | 0.2809 | 0.7127 | 0.8553 |
| n=500 | 0.3188 | 0.6803 | 0.8575 |
| n=2000 | 0.3327 | 0.6682 | 0.8555 |

**观察**：
- RSF 的性能平稳，不因样本量变化而波动（0.8553~0.8575）
- Weibull 在小样本时表现最好（0.7127），大样本时下降（0.6682）
- Cox 表现最差，但随样本量增加而缓慢改善（0.2809→0.3327）

---

### 1.3 模型性能按删失率的表现

| 删失率 | Cox | Weibull | RSF |
|--------|-----|---------|-----|
| 0.1（低删失） | 0.3206 | 0.6757 | 0.8429 |
| 0.3（中删失） | 0.3025 | 0.6961 | 0.8511 |
| 0.5（高删失） | 0.3093 | 0.6894 | 0.8743 |

**观察**：
- **Weibull**：删失率影响有限（0.6757~0.6961），性能相对稳定
- **RSF**：高删失（0.5）下表现最强（0.8743），非参数树模型对不完全信息的鲁棒性强
- **Cox**：删失率对性能影响不显著，整体表现最差

---

### 1.4 模型性能按协变量维度的表现（p 的效应）

| p维度 | Cox | Weibull | RSF |
|-------|-----|---------|-----|
| p=1（低维） | 0.4691 | 0.5346 | 0.7568 |
| p=50（高维） | 0.1525 | 0.8395 | 0.9554 |

**观察**：
- **Cox**：高维灾难（从0.47崩至0.15），线性假设完全失效
- **Weibull**：高维显著增强（从0.53增至0.84），参数模型在高维设置下优势明显
- **RSF**：高维优势最明显（从0.76跃至0.96），树模型天然适应高维特征

---

### 1.5 最差拟合场景分析

**最差拟合场景集中在**：n=100, p=50 组合
- Cox 在此情况下 C-index 接近随机（<0.15）
- Weibull 和 RSF 仍保持高准确性（>0.84）

**关键问题**：
- 小训练集 + 高维特征 + Cox 的线性假设导致完全崩塌
- 而树基和参数模型在同样条件下表现稳健

---

### 1.6 模型类型特性解析

#### Cox 比例风险模型
- **优点**：低维（p=1）时相对可用（C=0.47）
- **缺点**：
  - 全维度平均表现最差（C=0.31）
  - 高维（p=50）失效（C=0.15）
  - 线性假设对 Weibull 真实数据拟合不足
- **适用场景**：低维临床数据且期望不高

#### Weibull AFT 参数模型
- **优点**：
  - 高维表现优异（C=0.84）
  - 与数据生成过程的先验匹配
  - 整体均衡（C=0.687）
- **缺点**：
  - 小样本参数估计波动
  - CSA 中会出现极端宽度
- **适用场景**：有参数假设依据；高维设置

#### Random Survival Forest 非参数模型
- **优点**：
  - 全面最优（C=0.856）
  - 高维顶级表现（C=0.955）
  - 高删失下最稳健（C=0.874）
  - 小样本表现最佳（C=0.855）
- **缺点**：
  - 无参数解释能力
  - 计算复杂度高
- **适用场景**：追求预测准确性；复杂非线性关系

---

### 1.7 结论与启示

1. **模型选择应严格根据数据维度**：
   - p=1 时可用 Cox，但 RSF 仍优（0.47 vs 0.76）
   - p>10 时 Cox 不可用；Weibull/RSF 必选

2. **RSF 是最全能的选择**：在所有场景下表现最稳定且最佳（C=0.856）

3. **Weibull 在参数匹配时表现优异**，高维场景（C=0.84）可与 RSF 竞争

4. **后续 CSA 保形预测应基于**：
   - 点预测准确的模型（RSF 最佳）
   - 必须考虑模型的数值稳定性

In [5]:
# CSA 实验：针对 p=50 组测试基础模型的保形生存预测性能
csa_results = []

# 基础模型选择：KM、Cox、Weibull、RSF
base_models = ['km', 'cox', 'weibull', 'rsf']
alpha = 0.1

for i, (X, surv, time, event) in enumerate(datasets):
    cfg = configs[i]

    print(f"CSA 第 {i+1} 组: n={cfg['n']}, p={cfg['p']}, cens={cfg['cens_rate']}, hetero={cfg['hetero']}")
    X_train, time_train, event_train, X_cal, time_cal, event_cal, X_test, time_test, event_test = split_survival_data(
        X, time, event, test_size=0.2, cal_size=0.25, random_state=2026
    )

    kmf = fit_kaplan_meier(time_train, event_train)
    cox = fit_cox(X_train, time_train, event_train)
    weibull = fit_weibull(X_train, time_train, event_train)
    rsf = fit_rsf(X_train, time_train, event_train)

    model_map = {
        'km': kmf,
        'cox': cox,
        'weibull': weibull,
        'rsf': rsf
    }

    for model_type in base_models:
        model = model_map[model_type]
        lower, upper, q_value = fit_csa_intervals(
            model, X_cal, time_cal, event_cal, X_test,
            alpha=alpha, model_type=model_type
        )
        metrics = evaluate_interval_coverage(lower, upper, time_test, event_test)

        csa_results.append({
            'n': cfg['n'],
            'p': cfg['p'],
            'cens_rate': cfg['cens_rate'],
            'hetero': cfg['hetero'],
            'model_type': model_type,
            'alpha': alpha,
            'coverage': metrics['coverage'],
            'coverage_uncensored': metrics['coverage_uncensored'],
            'coverage_censored': metrics['coverage_censored'],
            'mean_width': metrics['mean_width'],
            'width_cv': metrics['width_cv'],
            'q_value': q_value
        })


# 保存 CSA 实验结果
df_csa = pd.DataFrame(csa_results)
df_csa.to_csv('与CSA结合效果.csv', index=False)
print('CSA 实验完成：', df_csa.shape[0], '条记录')
df_csa
# 简要汇总
print('\nCSA 汇总：按模型类型查看平均指标')
print(df_csa.groupby('model_type')[['coverage','mean_width','width_cv']].mean())

CSA 第 1 组: n=100, p=1, cens=0.1, hetero=False
CSA 第 2 组: n=100, p=1, cens=0.1, hetero=True
CSA 第 3 组: n=100, p=50, cens=0.1, hetero=False
CSA 第 4 组: n=100, p=50, cens=0.1, hetero=True
CSA 第 5 组: n=100, p=1, cens=0.3, hetero=False
CSA 第 6 组: n=100, p=1, cens=0.3, hetero=True
CSA 第 7 组: n=100, p=50, cens=0.3, hetero=False
CSA 第 8 组: n=100, p=50, cens=0.3, hetero=True
CSA 第 9 组: n=100, p=1, cens=0.5, hetero=False
CSA 第 10 组: n=100, p=1, cens=0.5, hetero=True
CSA 第 11 组: n=100, p=50, cens=0.5, hetero=False
CSA 第 12 组: n=100, p=50, cens=0.5, hetero=True
CSA 第 13 组: n=500, p=1, cens=0.1, hetero=False
CSA 第 14 组: n=500, p=1, cens=0.1, hetero=True
CSA 第 15 组: n=500, p=50, cens=0.1, hetero=False
CSA 第 16 组: n=500, p=50, cens=0.1, hetero=True
CSA 第 17 组: n=500, p=1, cens=0.3, hetero=False
CSA 第 18 组: n=500, p=1, cens=0.3, hetero=True
CSA 第 19 组: n=500, p=50, cens=0.3, hetero=False
CSA 第 20 组: n=500, p=50, cens=0.3, hetero=True
CSA 第 21 组: n=500, p=1, cens=0.5, hetero=False
CSA 第 22 组: n=500, p=1

## 2. CSA 保形预测性能总结

### 2.1 核心问题设定

**保形预测目标**：
- 目标覆盖率：α = 0.1，期望覆盖率 = 90%
- 评估维度：
  1. **覆盖率**（Coverage）：区间是否包含真实生存时间
  2. **未删失覆盖率**（未删失样本是否被覆盖）
  3. **删失覆盖率**（删失样本的下界条件是否满足）
  4. **区间宽度**（Efficiency，越小越好）
  5. **宽度稳定性**（Width CV）

---

### 2.2 各模型 CSA 平均性能表

| 模型 | 覆盖率 | 未删失 | 删失 | 平均宽度 | 宽度CV | 平均q值 |
|------|--------|--------|------|----------|--------|---------|
| **RSF** | 91.59% | 93.25% | 88.42% | 4.21 | 0.1924 | 2.54 |
| **Weibull** | 90.17% | 91.08% | 88.84% | 17.07 | 0.1331 | 14.26 |
| **KM** | 89.97% | 87.55% | 91.75% | 3.73 | 0.0000 | 2.16 |
| **Cox** | 88.41% | 91.09% | 84.37% | inf | 0.1276 | inf |

---

### 2.3 模型性能分析

#### 覆盖率与效率排名
1. **RSF** - 覆盖率最高（91.59%），宽度合理（4.21）✅ **综合最优**
   - 小样本表现特优（n=100: 95.42%）
   - 删失处理稳健（删失覆盖 88.42%）

2. **Weibull** - 覆盖率达标（90.17%），但宽度偏大（17.07）⚠️
   - 小样本严重问题：n=100 平均宽度 43.28（20倍于 RSF）
   - 低删失时宽度爆炸

3. **KM** - 覆盖率达标（89.97%），宽度极优（3.73） ✅
   - 宽度完全稳定（CV≈0）
   - 缺陷：无个体化，所有样本区间完全相同

4. **Cox** - 覆盖率最低（88.41%），频繁数值问题 ❌
   - n=100 出现 inf（数值不稳定）
   - 删失覆盖率下降（差 6.72%）

---

### 2.4 按样本量的 CSA 性能变化

#### RSF + CSA（最稳定）
```
n=100   → coverage 95.42%, width 3.99, q 2.40
n=500   → coverage 90.33%, width 4.35, q 2.59
n=2000  → coverage 89.02%, width 4.28, q 2.63
```
**观察**：小样本表现最优，性能稳定

#### Weibull + CSA（小样本爆炸）
```
n=100   → coverage 90.83%, width 43.28, q 33.63 ⚠️⚠️
n=500   → coverage 90.25%, width 4.14,  q  3.11
n=2000  → coverage 89.44%, width 3.78,  q  2.83
```
**关键问题**：n=100 时区间宽度爆炸（是 RSF 的 10 倍），最坏情况达 342.32

#### KM + CSA（宽度稳定）
```
n=100   → coverage 92.50%, width 3.79
n=500   → coverage 88.00%, width 3.68
n=2000  → coverage 89.42%, width 3.73
```
**观察**：宽度完全稳定，覆盖率变化不大

#### Cox + CSA（数值问题频繁）
```
n=100   → coverage 87.50%, width inf    ⚠️⚠️
n=500   → coverage 87.83%, width inf    ⚠️⚠️
n=2000  → coverage 89.90%, width 4.36
```
**问题**：仅 n=2000 可用，小样本完全失效

---

### 2.5 按删失率的 CSA 性能变化

#### 覆盖率表现（4 行 3 列）
| 模型 | cens=0.1 | cens=0.3 | cens=0.5 |
|------|----------|----------|----------|
| **RSF** | 91.67% | 92.06% | 91.04% |
| **Weibull** | 90.52% | 89.63% | 90.38% |
| **KM** | 91.63% | 89.19% | 89.10% |
| **Cox** | 88.83% | 88.19% | 88.21% |

#### 平均宽度表现
| 模型 | cens=0.1 | cens=0.3 | cens=0.5 |
|------|----------|----------|----------|
| **RSF** | 4.10 | 4.12 | 4.40 |
| **Weibull** | 5.52 | 10.73 | 34.95 ⚠️ |
| **KM** | 4.52 | 3.69 | 2.99 |
| **Cox** | inf | inf | inf |

**关键发现**：
- **Weibull 高度删失敏感**：低删失时极端（5.52），高删失时爆炸（34.95）
- **RSF 最稳定**：各删失率下覆盖率 91~92%，宽度 4.1~4.4
- **Cox 数值问题**：多数配置出现 inf 或接近 inf

---

### 2.6 删失样本覆盖率深度分析

#### 未删失 vs 删失覆盖率对比

| 模型 | 未删失覆盖 | 删失覆盖 | 覆盖差异 |
|------|-----------|---------|---------|
| **KM** | 87.55% | 91.75% | -4.20% ✓ |
| **RSF** | 93.25% | 88.42% | +4.83% |
| **Weibull** | 91.08% | 88.84% | +2.24% |
| **Cox** | 91.09% | 84.37% | +6.72% ⚠️ |

**关键问题**：
- **KM 最优**：删失覆盖反而更好，满足保形假设
- **Cox 最严重**：删失覆盖下降 6.72%，可靠性不足
- **RSF 可以接受**：差异 4.83%，在可控范围

---

### 2.7 最差 CSA 场景分析

#### 覆盖率最低的 5 个记录
```
1. Cox, n=100, p=50, cens=0.3, hetero=False  → 70% ⚠️⚠️⚠️
2. Cox, n=100, p=50, cens=0.3, hetero=True   → 75%
3. Cox, n=500, p=1, cens=0.5, hetero=True    → 76%
4. Cox, n=100, p=50, cens=0.1, hetero=False  → 80% (inf 宽度)
5. Weibull, n=100, p=50, cens=0.1, hetero=True → 80%
```

**模式**：
- **Cox 占据前 4 名最差**，高维情景尤其糟糕
- **Weibull 仅在特定条件下** (n=100, 低删失, 高维) 失效

#### 区间宽度极端值（排除 inf）

```
1. Weibull, n=100, cens=0.5, hetero=True   → 342.32 ⚠️⚠️⚠️
2. Weibull, n=100, cens=0.3, hetero=True   → 64.48
3. Weibull, n=100, cens=0.5, hetero=False  → 34.68
4. Weibull, n=100, cens=0.3, hetero=False  → 27.88
5. Weibull, n=100, cens=0.1, hetero=False  → 18.85
```

**观察**：
- **Weibull 的问题集中在 n=100**：全部前 5 个极端值都来自此
- **删失率低 + 异方差** 时最严重
- **原因**：小校准集中出现极端 Weibull 参数，导致分位数爆炸

---

### 2.8 CSA 方法的问题诊断

#### 问题 1：Cox 的数值稳定性崩塌（关键）
- **表现**：n≤500 时多数配置出现 inf
- **原因**：小样本下点预测波动极大，校准分数出现极值
- **影响**：基本不可用于小样本设置
- **建议**：需正则化或稳健校准方法

#### 问题 2：Weibull 的小样本极端性（关键）
- **表现**：n=100 平均宽度 43.28（vs RSF 3.99）
- **原因**：参数估计在小样本下不稳定，q 值被极端值主导
- **影响**：尽管覆盖率达标，但区间不实用
- **建议**：采用稳健分位数估计（IQR 截断）或加大校准集

#### 问题 3：RSF 的删失处理偏差（轻微）
- **表现**：删失覆盖率 88.42%，差未删失 4.83%
- **原因**：生存函数在删失点处乐观估计
- **影响**：不使用更严格条件可能低估真实风险
- **建议**：考虑删失特化的非一致性分数

---

### 2.9 CSA 模型效能评定与推荐

#### ✅ 强烈推荐（达到或超过 90% 覆盖率 + 宽度合理）
- **RSF+CSA**：覆盖 91.59%，宽度 4.21，综合最优，对所有样本量友好
- **KM+CSA**：覆盖 89.97%，宽度 3.73，效率最优但无个体化

#### ⚠️ 有条件推荐（存在特定缺陷）
- **Weibull+CSA**（仅 n≥500）：覆盖 90%+，宽度 3.8~4.1，可用性有限
- **Cox+CSA**（仅 n≥2000）：覆盖 89.90%，宽度 4.36，小样本不可用

#### ❌ 不推荐（存在系统性问题）
- **Weibull+CSA（n=100）**：宽度极端（43），不实用
- **Cox+CSA（n≤500）**：数值不稳定或覆盖不足

---

### 2.10 科学发现总结

1. **保形预测的稳定性高度依赖于点预测的数值可靠性**
   - 树模型（RSF）表现最稳定
   - 参数模型（Weibull）在小样本脆弱
   - 半参数模型（Cox）在小样本失效

2. **样本量与模型适用性的匹配**
   - n<500 时，仅 RSF 和 KM 可靠
   - n≥2000 时，所有模型基本可用（Cox 除外）

3. **删失率与方差异构对保形性的影响**
   - 高删失场景下 Weibull 的不稳定性更严重
   - 异方差条件下 Cox 的性能显著恶化

---

### 2.11 后续改进建议

- [ ] 对 Weibull 采用稳健分位数估计（IQR-based）以处理极端 q 值
- [ ] 对 RSF 设计删失特化的非一致性分数定义
- [ ] 对 Cox 添加正则化校准步骤以提升数值稳定性
- [ ] 增加校准集大小影响的系统评估